# EN4554 – Assignment 2: Convolutional Neural Networks

**Name:** John Praveen  &nbsp;&nbsp; **Index No:** `220491B` *(fill in)*  
**GitHub:** https://github.com/john-praveen/EN4554-Assignment-2

**Contents:** Setup · Task 1 Backpropagation · Task 2 Optimizers · Task 3 Regularization & residual connections · Task 4 Best CNN (optional)

## 0. Setup

This section covers the environment, the fixed seeds, and the CIFAR-10 data pipeline. CIFAR-10 is split 45k train / 5k validation, and each channel is normalised with the **training-split** mean and std. The test set is not touched until Task 4.

In [ ]:
import os, time, math, random
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision, torchvision.transforms as T
import matplotlib.pyplot as plt

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('torch', torch.__version__, '| device:', device,
      '|', torch.cuda.get_device_name(0) if device.type == 'cuda' else 'CPU')
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': .3})

In [ ]:
# --- Data: 45k train / 5k val / 10k test (test used only once, in Task 4) ---
root = './data'
full_train = torchvision.datasets.CIFAR10(root, train=True,  download=True)
test_set   = torchvision.datasets.CIFAR10(root, train=False, download=True)
CLASSES = full_train.classes

X_all = torch.from_numpy(full_train.data).permute(0, 3, 1, 2).float() / 255.   # N,3,32,32 in [0,1]
y_all = torch.tensor(full_train.targets)

g = torch.Generator().manual_seed(SEED)
perm = torch.randperm(len(X_all), generator=g)
tr_idx, va_idx = perm[:45000], perm[45000:]
X_tr, y_tr = X_all[tr_idx], y_all[tr_idx]
X_va, y_va = X_all[va_idx], y_all[va_idx]

# per-channel statistics from the TRAINING split only
MEAN = X_tr.mean(dim=(0, 2, 3), keepdim=True)
STD  = X_tr.std(dim=(0, 2, 3), keepdim=True)
norm = lambda x: (x - MEAN) / STD
X_tr, X_va = norm(X_tr), norm(X_va)

# test tensors are prepared here but ONLY used in Task 4
X_te = norm(torch.from_numpy(test_set.data).permute(0, 3, 1, 2).float() / 255.)
y_te = torch.tensor(test_set.targets)

# keep everything on the GPU: CIFAR-10 is tiny, so this removes data-loader overhead
X_tr, y_tr, X_va, y_va = [t.to(device) for t in (X_tr, y_tr, X_va, y_va)]
print('train', tuple(X_tr.shape), 'val', tuple(X_va.shape), 'test', tuple(X_te.shape))
print('mean', MEAN.flatten().tolist(), '
std ', STD.flatten().tolist())

In [ ]:
# --- Shared helpers used by every task ---
def batches(X, y, bs, shuffle=True, drop_last=False):
    n = len(X)
    idx = torch.randperm(n, device=X.device) if shuffle else torch.arange(n, device=X.device)
    for i in range(0, n, bs):
        j = idx[i:i + bs]
        if drop_last and len(j) < bs: break
        yield X[j], y[j]

@torch.no_grad()
def evaluate(model, X, y, bs=1000):
    model.eval()
    loss = correct = 0
    for xb, yb in batches(X, y, bs, shuffle=False):
        out = model(xb.to(device))
        loss    += F.cross_entropy(out, yb.to(device), reduction='sum').item()
        correct += (out.argmax(1) == yb.to(device)).sum().item()
    return loss / len(X), correct / len(X)

def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

# sanity check: class balance and a few images
print('train class counts:', torch.bincount(y_tr.cpu()).tolist())
fig, ax = plt.subplots(1, 8, figsize=(12, 2))
for i, a in enumerate(ax):
    img = (X_tr[i].cpu() * STD[0] + MEAN[0]).permute(1, 2, 0).clamp(0, 1)
    a.imshow(img); a.set_title(CLASSES[y_tr[i]], fontsize=8); a.axis('off')
plt.show()

**Setup notes.** Everything is held on the GPU as tensors and a hand-written batch iterator is used, so no `DataLoader` is needed. This keeps the many training runs in Tasks 2–4 fast on a T4. The normalisation statistics come from the 45k training images only, so no information from validation or test leaks into training. Seeds are fixed through `set_seed()`, which is called again at the start of each experiment.